In [ ]:
### loss 계산 (1개 배치)

def calc_loss_batch( input_batch, target_batch, model, device ) :
    ## 하나의 배치에 대한 loss 계산

    # 1. input, target 을 GPU 로 이동
    input_batch, target_batch = input_batch.to(device), target_batch.to(device)     # [B, N]

    # 2. 순전파 forward 진행
    logits = model(input_batch)     # [B, N, vocab_size]

    # 3. loss 계산 (CrossEntropyLoss)
    #   - CrossEntropyLoss 는 (N, C) 형태의 입력이 필요 N : # of token, C : vocab_size = # of classes
    #   - [B, N, C] -> [B * N, C] 형태로 변환 필요 => flatten(0, 1)
    loss = torch.nn.functional.cross_entropy( logits.flatten(0, 1), target_batch.flatten() )

    return loss

In [ ]:
### loss 계산 (모든 데이터 셋)

def calc_loss_loader( data_loader, model, device, num_batches = None ) :
    total_loss = 0

    if len(data_loader) == 0:
        return float("nan")
    elif num_batches is None:
        num_batches = len(data_loader)
    else:
        num_batches = min(num_batches, len(data_loader))

    # loss 계산
    for i, (input_batch, target_batch) in enumerate(data_loader) :
        if i < num_batches:
            loss = calc_loss_batch( input_batch, target_batch, model, device )
            total_loss += loss.item()   # tensor 에서 float 실수 값만 추출해서 누적
        else:
            break

    return total_loss / num_batches

In [ ]:
### model train

def train_model_simple(model, train_loader, val_loader, optimizer, device, num_epochs, eval_freq, eval_iter, start_context, tokenizer):
    ## 모델을 train/val dataset 으로 학습

    # for logging
    train_losses, val_losess, track_tokens_seen = [], [], []
    tokens_seen = 0
    global_step = 1

    # num_epochs 만큼 학습
    for epoch in range(num_epochs) :
        # 1. train mode 설정
        model.train()

        # 2. dataloader 로 train data 을 배치만큼 읽어와서 학습
        for input_batch, target_batch in train_loader :
            # 2-1. optimizer 기울기 초기화
            optimizer.zero_grad()

            # 2-2. 순전파 forward 진행해서 loss 계산
            loss = calc_loss_batch(input_batch, target_batch, model, device)

            # 2-3. 역전파 back-propagation : 각 파라미터 기울기 계산
            loss.backward()

            # 2-4. 가중치 업데이트 : 기울기로 파라미터 업데이트
            optimizer.step()

            tokens_seen += input_batch.numel()
            global_step += 1

            # 일정 스텝마다 로깅
            if global_step % eval_freq == 0 :
                train_loss, val_loss = evaluate_model( model, train_loader, val_loader, device, eval_iter )
                train_losses.append(train_loss)
                val_losses.append(val_loss)
                track_tokens_seen.append(tokens_seen)
                print(f"Ep {epoch+1} (Step {global_step:06d}) : "
                       "Train loss {trainloss:.3f}, Val loss {val_loss:.3f}")
            
        # 한 epoch 학습 후 샘플 문장 테스트
        generate_and_print_sample( model, tokenizer, device, start_context )

    return train_losses, val_losess, track_tokens_seen

In [ ]:
### 모델 평가 함수

def evaluate_model( mode, train_loader, val_loader, device, eval_iter ) :
    ## 현재 모델의 성능을 train/val 데이터 셋으로 평가

    # 1. 평가 모드로 설정
    model.eval()

    # 2. loss 계산
    with torch.no_grad():
        train_loss = calc_loss_loader( train_loader, model, device, num_batches=eval_iter )
        val_loss = calc_loss_loader( val_loader, model, device, numb_batches=eval_iter )

    # 3. 평가 종로 후 train 모드로 설정
    model.train()

    return train_loss, val_loss

In [ ]:
### 모델을 이요한 결과 텍스트 생성

def generate( model, idx, max_new_tokens, context_size, eos_id=None ):
    
    # max_new_tokens 길이 만큼 생성
    for _ in range(max_new_tokens):
        # 1. 문맥 자르기
        idx_cond = idx[:, -context_size:]   # 최대 컨텍스트 길이 넘지 않도록 클리핑

        # 2. 모델 예측
        with torch.no_grad():
            logits = model( idx_cond )      # [B, N, C]

        # 3. 다음 단어 예측
        logits = logits[:, -1, :]   # 마지막 위치의 logits 만 사용

        # 4. 가장 높은 확률을 가지는 토큰 선택
        idx_next = torch.argmax(logits, dim=-1, keepdim=True)

        if idx_next == eos_id:
            break

        # 5. 생성된 문장 이어 붙이기
        idx = torch.cat((idx, idx_next), dim=1)

    return idx

In [ ]:
### text <-> token 유틸리티 함수

def text_to_token_ids( text, tokenizer ) :
    encoded = tokenizer.encode( text )
    encoded_tensor = torch.tensor( encoded ).unsqueeze( 0 )     # 단일 시퀀스를 텐서로 만들고, 배치 차원 추가
    return encoded_tensor

def token_ids_to_text( token_ids, tokenizer ) :
    flat = token_ids.squeeze( 0 )                               # 배치 차원을 제거 [[ 12, 423, 23 ]] -> [ 12, 423, 23 ]
    return tokenizer.decode( flat.tolist() )

In [ ]:
### 학습 및 모델 저장/로드

def main( gpt_config, settings ) :
    # 랜덤 시드 고정 및 device 설정
    torch.manual_seed(123)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    ## 1. 데이터 준비
    file_path = "datas/the-verdict.txt"
    url = "..."

    if not os.path.exists( file_path ):
        response = requests.get( url, timeout=30 )
        response.raise_for_status()
        text_data = response.text
        
        with open(file_path, "w", encoding="utf-8") as f:
            f.write(text_data)
    else:
        with open(file_path, "r", encoding="utf-8") as f:
            text_data = f.read()

    ## 2. 모델 생성 및 옵티마이저 초기화
    model = GPTModel( gpt_config )
    model.to(device)

    optimizer = torch.optim.AdamW( model.parameters(), lr=settings["learning_rate"], weight_decay=settings["weight_decay"] )

    ## 3. 데이터 로더 생성
    train_ratio = 0.9
    split_idx = int( train_ratio * len(text_data) )

    ## 3-1. train data set
    train_loader = create_dataloader_v1(
        text_data[ :split_idx ],
        batch_size=settings["batch_size"]
        max_length=gpt_config["context_length"]
        stride=gpt_config["context_length"]
        shuffle=True,       # Shuffle / drop_last True
        drop_last=True,
        num_workers=0
    )

    ## 3-2. val data set
    val_loader = create_dataloader_v1(
        text_data[ split_idx: ],
        batch_size=settings["batch_size"]
        max_length=gpt_config["context_length"]
        stride=gpt_config["context_length"]
        shuffle=False,      # Shuffle / drop_last False
        drop_last=False,
        num_workers=0
    )

    ## 4. 학습 시작
    tokenizer = tiktoken.get_encoding("gpt2")

    train_losses, val_losses, tokens_seen = train_model_simple( model, train_loader, val_loader, optimizer, device,
        num_epochs=settings["num_epochs"], eval_freq=5, eval_iter=1, 
        start_context="Every effort moves you", tokenizer=tokenizer )

    return train_losses, val_losses, tokens_seen, model


=================================================================
### main 함수 호출

# GPT-2 Small (124M) 모델 설정
GPT_CONFIG_124M = {
    "vocab_size": 50257,    # 어휘 크기
    "context_length": 256,  # 훈련 속도를 위해 원본(1024)보다 줄임
    "emb_dim": 768,         # 임베딩 벡터 차원
    "n_heads": 12,          # 어텐션 헤드 개수
    "n_layers": 12,         # 레이어 깊이
    "drop_rate": 0.1,       # 과적합 방지 드롭아웃
    "qkv_bias": False       
}

# 학습 하이퍼파라미터
OTHER_SETTINGS = {
    "learning_rate": 5e-4, 
    "num_epochs": 10,       
    "batch_size": 2,        
    "weight_decay": 0.1     
}

train_losses, val_losses, tokens_seen, model = main( GPT_CONFIG_124M, OTHER_SETTINGS )

## 학습 결과 모델 저장
torch.save( model.state_dict(), "output/model.pth" )

## 학습 결과 로드해서 모델 초기화 
model = GPTModel( GPT_CONFIG_124M )
model.load_state_dict( torch.load("output/model.pth"), weights_only=True )
model.to(device)